# OmniBench -> multi-representation cache (omni_v1)

Converts the OneData-OmniBench episodes (Lee + Lazizbek shards on Google Drive) into **one compact pickle** that the
multi-representation adaptor notebook (`multirep_robot.ipynb`) trains from. Runs on a **CPU** Colab runtime, no GPU needed.

**What it does**
1. Finds every episode in the shard folders and keeps only those whose `_COMPLETE.json` file sizes match and whose `validation.json` passes.
2. Builds the 4 main representations per sampled frame, plus 2 extra proxy signals:

| representation | built from | shape |
|---|---|---|
| view | RGB of front / overhead / side, resized to square | (3, R, R, 3) uint8 |
| point | depth + RGB + calibration of all 3 cameras, table and background removed, colored | (768, 3) xyz + (768, 3) rgb |
| proprio | 7 joints + mean finger opening + EE xyz, last 12 frames (1 s) | (12, 11) |
| speech | spoken instruction, 16 kHz log-mel | (250, 40) |
| tactile *(extra)* | 2-pad pressure **proxy**, last 12 frames | (12, 2, 16, 16) float16 |
| impact *(extra)* | synthetic impact audio, log-mel of the last 1.5 s **up to time t** | (75, 40) |

Not used as inputs: force (always 0: grasping is a scripted attachment), segmentation (only used to drop the table
from the point cloud), optical flow (front only, estimated from RGB), the 240 Hz physics log (simulator-internal).

3. Defines **frame-level questions** whose answers come from the saved simulator state (not from an LLM), samples them
balanced over answers, and stores the EE-action labels of the same frame (next expert EE delta + next gripper command).
4. Reports **shortcut baselines**: how well the answer can be guessed from the task family alone, the time in the
episode alone, or both. A question that these guess well does not need perception.

The canonical episode-level QA v1 (8 questions per episode) is kept in the cache and gets the same shortcut report.

## 1. Setup

In [ ]:
import os, sys, json, glob, time, zlib, pickle, datetime
from collections import Counter
from math import gcd
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
from PIL import Image

IN_COLAB = os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## 2. Configuration

**Two ways to get the 800-episode cache**
- *Raw shards*: one person reads both shard folders (below) and builds the cache. Needs access to both ~3 GB folders.
- *Per-shard caches, then merge* (no raw data exchanged): each person builds a cache of their own shard with
  `shard_roots=[own folder]`, `shard_share=0.5` and their own `out_path` (e.g. `omni_v1_cache_Lee.pkl`), and shares that
  file (~0.5 GB). One person then sets `merge_from=[both cache paths]` and runs the notebook again: sections 3-6 are
  skipped, the caches are concatenated (examples, episodes; nothing re-sampled), and sections 7-9 report and save the
  merged cache. `shard_share=0.5` makes the merged cache the same size as one built from both shards at once; every
  question stays balanced over answers within each shard.

Lazizbek's shard: open the shared folder in Google Drive -> right-click -> **Organize -> Add shortcut** -> My Drive /
`OneData_OmniBench_v1`. It then shows up next to your own `Lee` folder. A missing shard is skipped with a warning,
so the notebook also runs on one shard.

In [ ]:
BASE = "/content/drive/MyDrive/OneData_OmniBench_v1"
CFG = dict(
    shard_roots=[f"{BASE}/Lee", f"{BASE}/Lazizbek"],
    out_path=f"{BASE}/cache/omni_v1_cache.pkl",
    version="omni_v1",
    seed=0,
    view_res=128,               # px, square (source 192x144). 96 makes the cache ~45% smaller
    n_points=768,               # sampled from the 3 cameras together
    proprio_win=12,             # frames = 1 s at 12 FPS (same as the robot notebook)
    ahead=6,                    # next_action looks 0.5 s ahead; motion uses the last 0.5 s
    n_mels=40, speech_frames=250,
    impact_win_s=1.5, tactile_win=12,
    n_per_task={"train": 1200, "val": 150, "test_iid": 150, "test_compositional": 150, "test_visual_ood": 150},
    time_bin=6,                 # frames (0.5 s): time unit of the shortcut baselines
    shard_share=1.0,            # fraction of n_per_task to sample: 0.5 when each person builds a cache of their own shard
    merge_from=[],              # cache paths to merge instead of building from raw shards (sections 3-6 are skipped)
    goal_dist_m=0.33,           # goal_dist threshold: about the median object-goal distance of the generator (0.20-0.44 m)
)
if os.environ.get("OMNI_ROOTS"):                     # local test runs
    CFG["shard_roots"] = os.environ["OMNI_ROOTS"].split(os.pathsep)
    CFG["out_path"] = os.environ.get("OMNI_OUT", "omni_v1_cache.pkl")
if os.environ.get("OMNI_N"):
    CFG["n_per_task"] = {k: int(os.environ["OMNI_N"]) for k in CFG["n_per_task"]}
CFG["n_per_task"] = {k: max(1, int(round(v * CFG["shard_share"]))) for k, v in CFG["n_per_task"].items()}
if os.environ.get("OMNI_MERGE"): CFG["merge_from"] = os.environ["OMNI_MERGE"].split(os.pathsep)
MERGE = bool(CFG["merge_from"])
SPLITS = list(CFG["n_per_task"])
FPS, SR, CAMS = 12, 16000, ("front", "overhead", "side")
PC_CENTER, PC_SCALE = np.array([0.45, 0.0, 0.20]), 0.40      # same point normalization as multirep_robot
print(json.dumps({k: v for k, v in CFG.items() if k != "n_per_task"}, indent=1))

## 3. Find and audit the episodes

In [ ]:
if not MERGE:   # building from raw shards (sections 3-6); merging: see 6b
    def check_episode(d):
        """(ok, reason). Same size check as the generator's audit_saved_shard, plus validation.json."""
        marker = os.path.join(d, "_COMPLETE.json")
        if not os.path.exists(marker): return False, "no _COMPLETE.json"
        for name, rec in json.load(open(marker))["sha256"].items():
            p = os.path.join(d, name)
            if not os.path.exists(p) or os.path.getsize(p) != rec["bytes"]: return False, "missing / bad size: " + name
        v = os.path.join(d, "validation.json")
        if os.path.exists(v) and not json.load(open(v)).get("all_pass", False): return False, "validation failed"
        return True, ""

    def find_episodes(roots):
        good, bad = [], []
        for r in roots:
            if not os.path.isdir(r): print("WARNING shard folder not found, skipped:", r); continue
            dirs = sorted(glob.glob(os.path.join(r, "episodes", "T*_*"))) or sorted(glob.glob(os.path.join(r, "T*_*")))
            for d in dirs:
                ok, why = check_episode(d)
                (good.append(d) if ok else bad.append((d, why)))
            print(f"{r}: {len(dirs)} episode folders")
        ids = [os.path.basename(d) for d in good]
        assert len(ids) == len(set(ids)), "the same episode id appears in two shards"
        return good, bad

    EP_DIRS, BAD = find_episodes(CFG["shard_roots"])
    print(f"{len(EP_DIRS)} complete episodes, {len(BAD)} rejected")
    for d, why in BAD[:20]: print("  rejected", os.path.basename(d), "-", why)
    assert EP_DIRS, "no episodes found: check CFG['shard_roots']"

## 4. Per-episode signals (light pass)

Everything except the camera arrays: robot / object state, phases, actions, speech, tactile and impact audio.
The heavy camera file is read only for the frames that are sampled (section 6).

In [ ]:
if not MERGE:   # building from raw shards (sections 3-6); merging: see 6b
    def _mel_fb(n_fft=400, n_mels=CFG["n_mels"]):
        hz2mel, mel2hz = lambda f: 2595 * np.log10(1 + f / 700), lambda m: 700 * (10 ** (m / 2595) - 1)
        bins = np.floor((n_fft + 1) * mel2hz(np.linspace(0, hz2mel(SR / 2), n_mels + 2)) / SR).astype(int)
        fb = np.zeros((n_mels, n_fft // 2 + 1))
        for m in range(n_mels):
            l, c, r = bins[m:m + 3]
            fb[m, l:c] = (np.arange(l, c) - l) / max(c - l, 1); fb[m, c:r] = (r - np.arange(c, r)) / max(r - c, 1)
        return fb
    MEL_FB, N_FFT, HOP = _mel_fb(), 400, 320                  # 25 ms window, 20 ms hop (as in multirep_robot)
    LOG0 = np.float32(np.log(1e-6))

    def log_mel(x):
        """(n_frames, n_mels) log-mel of a 16 kHz waveform; frame i covers samples [i*HOP, i*HOP + N_FFT)."""
        x = np.pad(x, (0, max(0, N_FFT - len(x))))
        n = 1 + (len(x) - N_FFT) // HOP
        fr = np.lib.stride_tricks.as_strided(x, (n, N_FFT), (x.strides[0] * HOP, x.strides[0])) * np.hanning(N_FFT)
        return np.log(np.abs(np.fft.rfft(fr, axis=1)) ** 2 @ MEL_FB.T + 1e-6).astype(np.float32)

    def read_wav16k(path):
        sr, x = wavfile.read(path)
        x = x.astype(np.float32) / (32768.0 if x.dtype == np.int16 else 1.0)
        if x.ndim > 1: x = x.mean(1)
        if sr != SR: g = gcd(SR, sr); x = resample_poly(x, SR // g, sr // g).astype(np.float32)
        return x

    def fixed_len(m, n):
        out = np.full((n, m.shape[1]), LOG0, np.float32); out[:min(n, len(m))] = m[:n]; return out

    def load_light(d):
        J = lambda f: json.load(open(os.path.join(d, f), encoding="utf8"))
        t, sc = J("task.json"), J("scene_and_replay.json")
        a, s = np.load(os.path.join(d, "actions.npz")), np.load(os.path.join(d, "robot_object_state.npz"))
        tac = np.load(os.path.join(d, "tactile_proxy.npz"))["pressure_2pads_PROXY"].astype(np.float16)
        ep = dict(id=t["episode_id"], dir=d, owner=t.get("owner"), split=t["split"], task_id=t["task_id"],
                  task_name=t["task_name"].replace("_", " "), instruction=t["instruction"],
                  color=t["style"]["target_name"], goal_color=t["style"]["goal_name"],
                  start=np.array(t["initial_xyz"], np.float32), goal=np.array(t["target_xyz"], np.float32),
                  edge=float(t["cube_edge_m"]), seed=t["seed"],
                  ts=a["timestamp_s"], T=len(a["timestamp_s"]), phase=a["expert_phase"].astype(str),
                  q=s["joint_position_rad"].astype(np.float32), finger=s["finger_position_m"].mean(1).astype(np.float32),
                  ee=s["ee_position_world_m"].astype(np.float32), obj=s["object_position_world_m"].astype(np.float32),
                  attached=s["scripted_attachment_active"].astype(bool),
                  ee_delta=a["expert_next_ee_delta_from_current_world_m"].astype(np.float32),
                  ee_target=a["expert_next_ee_target_world_m"].astype(np.float32),
                  grip_next=a["gripper_next_command_m"].astype(np.float32), action_valid=a["action_valid"].astype(bool),
                  qa_v1=[json.loads(l) for l in open(os.path.join(d, "qa.jsonl"), encoding="utf8")],
                  calib=sc["camera_calibration"], bodies=sc["object_body_ids"])
        ep["mel"] = fixed_len(log_mel(read_wav16k(os.path.join(d, "spoken_instruction.wav"))), CFG["speech_frames"])
        ep["_impact"] = log_mel(read_wav16k(os.path.join(d, "impact_audio_SYNTHETIC.wav")))   # dropped after section 6
        ep["_tactile"] = tac
        assert len(ep["q"]) == ep["T"] == len(tac)
        return ep

    t0 = time.time(); EPS = {sp: [] for sp in SPLITS}
    for i, d in enumerate(EP_DIRS):
        ep = load_light(d)
        if ep["split"] not in EPS: print("WARNING unknown split", ep["split"], ep["id"]); continue
        EPS[ep["split"]].append(ep)
        if (i + 1) % 100 == 0: print(f"  {i + 1}/{len(EP_DIRS)} ({time.time() - t0:.0f}s)")
    print(f"light pass {time.time() - t0:.0f}s:", {sp: len(v) for sp, v in EPS.items()})
    from collections import Counter
    for sp, eps in EPS.items():
        if eps: print(f"  {sp:>18}: tasks {dict(sorted(Counter(e['task_id'] for e in eps).items()))}")

## 5. Frame-level questions

The expert's waypoints are **fixed relative to the object** (the EE always starts 9 cm behind and 28 cm above it) and
follow a **fixed time schedule**, so anything about the gripper-object relation or the current phase is a function
of time (and proprio). The questions therefore ask about what *differs between episodes*: the task, the object's
color, and where the goal is relative to the object (asked only before the object starts moving, so the robot's own
motion does not reveal it). `motion` and `next_action` are kept as proprio-side questions, as in the robot notebook.

| question | answer | expected representation |
|---|---|---|
| task | pick place / stack / press button / drag pull / push slide / ... | speech |
| color | blue / orange / green (magenta in visual OOD) | speech, view, point |
| motion | up / down / left / right / forward / backward | proprio |
| next_action | next expert phase in 0.5 s | proprio + speech |
| goal_side | closer / farther (goal vs object, distance from the robot base); T1/T3/T4/T7 | view, point |
| goal_dist | yes / no (goal within 33 cm of the object, `goal_dist_m`); T1/T3/T4/T7 | point, view |

From the generator's sampling ranges: in train/val/IID about 34% of goal_side answers are "farther" and goal_dist is
about 50/50 (balanced sampling evens both out). In **test_compositional** the goal is moved next to the start, so
goal_side is always "closer" and goal_dist always "yes": a pure distribution-shift test.

Directions are in the robot frame: forward = +x (away from the base), left = +y, up = +z.

In [ ]:
QUESTIONS = {
    "task": "What task is the robot doing?",
    "color": "What color is the object the robot must handle?",
    "motion": "Which way is the gripper moving?",
    "next_action": "What will the robot do next?",
    "goal_side": "Is the goal closer to the robot or farther away than the object?",
    "goal_dist": f"Is the goal within {CFG['goal_dist_m'] * 100:.0f} cm of the object?",
}
TASKS = {"task": "speech", "color": "speech / view / point", "motion": "proprio", "next_action": "proprio + speech",
         "goal_side": "view / point", "goal_dist": "point / view"}
MOTIONS = {"up": (2, 1), "down": (2, -1), "left": (1, 1), "right": (1, -1), "forward": (0, 1), "backward": (0, -1)}
AHEAD = CFG["ahead"]


if not MERGE:   # building from raw shards (sections 3-6); merging: see 6b
    def candidates(ep):
        """All (task, frame, answer, (primary, secondary)) of one episode; sampling balances over primary, then secondary."""
        T, ee, out = ep["T"], ep["ee"], []
        g, s = ep["goal"][:2], ep["start"][:2]
        still = np.linalg.norm(ep["obj"][:, :2] - s, axis=1) < 0.01          # object has not moved yet
        pre = [k for k in range(T) if still[k] and not ep["attached"][k]]
        dx, dist = float(g[0] - s[0]), float(np.linalg.norm(g - s))
        for k in range(T):
            out.append(("task", k, ep["task_name"], (ep["task_name"], None)))
            out.append(("color", k, ep["color"], (ep["color"], ep["task_id"])))
            if k >= AHEAD:
                d = ee[k] - ee[k - AHEAD]
                if np.linalg.norm(d) > 0.03:
                    ax = int(np.abs(d).argmax()); a = [m for m, (i, sg) in MOTIONS.items() if i == ax and sg == np.sign(d[ax])][0]
                    out.append(("motion", k, a, (a, None)))
            if k + AHEAD < T:
                nxt = ep["phase"][k + AHEAD]; out.append(("next_action", k, nxt, (nxt, bool(nxt != ep["phase"][k]))))
        for k in pre:
            if abs(dx) > 0.02:                                               # T2/T8 move along y only, T5/T6 stay
                a = "farther" if dx > 0 else "closer"; out.append(("goal_side", k, a, (a, ep["task_id"])))
            if abs(dx) > 1e-4 and abs(dist - CFG["goal_dist_m"]) > 0.02:     # 2 cm margin; T2/T8 (goal straight along y,
                                                                              # 96% "yes") and T5/T6 (goal = start) excluded
                a = "yes" if dist < CFG["goal_dist_m"] else "no"; out.append(("goal_dist", k, a, (a, ep["task_id"])))
        return out

    def balanced(groups, n, rng):
        """{primary: {secondary: [items]}} -> n items, round-robin over primary, then secondary (reuse only when a group runs out)."""
        prim = sorted(groups, key=str); sec = {k: sorted(groups[k], key=str) for k in prim}
        shuf = {(a, b): [groups[a][b][i] for i in rng.permutation(len(groups[a][b]))] for a in prim for b in sec[a]}
        cnt_p, cnt_s, out = {a: 0 for a in prim}, {key: 0 for key in shuf}, []
        for j in range(n):
            a = prim[j % len(prim)]; b = sec[a][cnt_p[a] % len(sec[a])]; cnt_p[a] += 1
            grp = shuf[a, b]; out.append(grp[cnt_s[a, b] % len(grp)]); cnt_s[a, b] += 1
        return out

    PICKS = {}
    for si, sp in enumerate(SPLITS):
        rng = np.random.default_rng([CFG["seed"], si]); cand = {}
        for e, ep in enumerate(EPS[sp]):
            for task, k, a, (k1, k2) in candidates(ep):
                cand.setdefault(task, {}).setdefault(k1, {}).setdefault(k2, []).append((e, k, a))
        PICKS[sp] = []
        if not EPS[sp]: continue
        for task in QUESTIONS:
            if task not in cand: print(f"  {sp}: no candidates for {task} (skipped)"); continue
            PICKS[sp] += [(task, e, k, a) for e, k, a in balanced(cand[task], CFG["n_per_task"][sp], rng)]
        if PICKS[sp]:
            n_frames = len({(e, k) for _, e, k, _ in PICKS[sp]})
            print(f"{sp:>18}: {len(PICKS[sp])} examples from {len(EPS[sp])} episodes, {n_frames} distinct frames")

## 6. Build the examples (heavy pass: camera arrays of the sampled frames only)

In [ ]:
if not MERGE:   # building from raw shards (sections 3-6); merging: see 6b
    def unproject(cal, depth_m, ys, xs):
        """Pixel + linear depth -> world xyz, same convention as the generator (OpenGL clip space, column-major matrices)."""
        W, H, near, far = cal["width"], cal["height"], cal["near_m"], cal["far_m"]
        V = np.asarray(cal["view_col_major"]).reshape(4, 4, order="F"); P = np.asarray(cal["projection_col_major"]).reshape(4, 4, order="F")
        inv = np.linalg.inv(P @ V)
        z = depth_m[ys, xs]; zb = (far - far * near / z) / (far - near)
        clip = np.stack([2 * (xs + .5) / W - 1, 1 - 2 * (ys + .5) / H, 2 * zb - 1, np.ones_like(z)], -1)
        w = clip @ inv.T
        return w[:, :3] / w[:, 3:]

    def frame_inputs(ep, k, cams, rng):
        """views (3, R, R, 3) uint8, points (N, 3) normalized, point_rgb (N, 3) in [0, 1]."""
        R, views, pts, rgbs = CFG["view_res"], [], [], []
        plane = 0                                                     # loaded first by the generator; not in object_body_ids
        assert plane not in ep["bodies"].values()
        for c in CAMS:
            rgb = cams[c + "_rgb"][k]
            views.append(np.asarray(Image.fromarray(rgb).resize((R, R), Image.BILINEAR)))
            cal = ep["calib"][c]; d = cams[c + "_depth_mm"][k].astype(np.float64) / 1000
            seg = cams[c + "_segmentation_int32"][k]; body = np.where(seg < 0, -1, seg & ((1 << 24) - 1))
            ys, xs = np.where((body >= 0) & (body != plane) & (d > cal["near_m"]) & (d < cal["far_m"] - 0.05))
            pts.append(unproject(cal, d, ys, xs)); rgbs.append(rgb[ys, xs] / 255.0)
        pts, rgbs = np.concatenate(pts), np.concatenate(rgbs)
        idx = rng.choice(len(pts), CFG["n_points"], replace=len(pts) < CFG["n_points"])
        return np.stack(views), ((pts[idx] - PC_CENTER) / PC_SCALE).astype(np.float32), rgbs[idx].astype(np.float32)

    IMPACT_W = int(round(CFG["impact_win_s"] * SR / HOP))
    def causal_window(ep, k):
        """Proprio / tactile over the last proprio_win frames (left-padded with frame 0), impact log-mel up to time t."""
        W = CFG["proprio_win"]; lo = max(0, k - W + 1)
        win = np.r_[[0] * (W - (k - lo + 1)), np.arange(lo, k + 1)].astype(int)
        proprio = np.c_[ep["q"][win], ep["finger"][win], ep["ee"][win]].astype(np.float32)          # (W, 11)
        tactile = ep["_tactile"][win[-CFG["tactile_win"]:]]
        end = max(0, int((ep["ts"][k] * SR - N_FFT) // HOP) + 1)       # mel frames that end before t
        m = ep["_impact"][max(0, end - IMPACT_W):end]
        impact = np.full((IMPACT_W, m.shape[1]), LOG0, np.float32); impact[IMPACT_W - len(m):] = m
        return proprio, tactile, impact

    DATA = {}; t0 = time.time()
    for sp in SPLITS:
        need = {}
        for task, e, k, a in PICKS[sp]: need.setdefault(e, set()).add(k)
        frames = {}
        for j, (e, ks) in enumerate(sorted(need.items())):
            ep = EPS[sp][e]
            with np.load(os.path.join(ep["dir"], "multiview_rgb_depth_seg_flow.npz")) as z:
                cams = {f"{c}_{x}": z[f"{c}_{x}"] for c in CAMS for x in ("rgb", "depth_mm", "segmentation_int32")}
            for k in sorted(ks):
                rng = np.random.default_rng([CFG["seed"], zlib.crc32(ep["id"].encode()), k])
                frames[e, k] = frame_inputs(ep, k, cams, rng) + causal_window(ep, k)
            if (j + 1) % 50 == 0: print(f"  {sp}: {j + 1}/{len(need)} episodes ({time.time() - t0:.0f}s)")
        data = []
        for task, e, k, a in PICKS[sp]:
            ep = EPS[sp][e]; views, pts, prgb, proprio, tactile, impact = frames[e, k]
            data.append(dict(task=task, q=QUESTIONS[task], a=str(a), ep=e, k=k, episode_id=ep["id"], task_id=ep["task_id"],
                             views=views, points=pts, point_rgb=prgb, proprio=proprio, speech=ep["mel"],
                             tactile=tactile, impact=impact,
                             ee_delta=ep["ee_delta"][k], grip_next=np.float32(ep["grip_next"][k]),
                             action_valid=bool(ep["action_valid"][k])))
        rng = np.random.default_rng([CFG["seed"], 99, SPLITS.index(sp)])
        DATA[sp] = [data[i] for i in rng.permutation(len(data))]
        if data: print(f"{sp:>18}: {len(data)} examples, {len(frames)} frames ({time.time() - t0:.0f}s)")
    for eps in EPS.values():
        for ep in eps: ep.pop("_impact", None); ep.pop("_tactile", None)

## 6b. Merge per-shard caches (only with `merge_from`)

Concatenates the examples and episodes split by split (example `ep` indices are shifted to the merged episode list),
checks that the caches were built with the same version, questions and input settings, and that no episode appears
twice. Arrays shared between examples of one frame stay shared.

In [ ]:
if MERGE:
    import pickle
    SAME = ("version", "view_res", "n_points", "proprio_win", "ahead", "n_mels", "speech_frames", "impact_win_s",
            "tactile_win", "goal_dist_m", "seed")
    EPS, DATA, BAD, SOURCES = {sp: [] for sp in SPLITS}, {sp: [] for sp in SPLITS}, [], []
    ref = None
    for path in CFG["merge_from"]:
        c = pickle.load(open(path, "rb")); print(f"{path}: created {c['created']}, " +
              ", ".join(f"{sp} {len(v['data'])}/{len(v['eps'])}" for sp, v in c["splits"].items()))
        if ref is None: ref = c
        bad = [k for k in SAME if c["cfg"].get(k) != ref["cfg"].get(k)]
        assert not bad and c["questions"] == ref["questions"] and list(c["cams"]) == list(ref["cams"]),             f"{path} was built with other settings: {bad or 'questions / cameras'}"
        for sp in SPLITS:
            off = len(EPS[sp]); EPS[sp] += c["splits"][sp]["eps"]
            DATA[sp] += [dict(ex, ep=ex["ep"] + off) for ex in c["splits"][sp]["data"]]
        BAD += [tuple(r) for r in c.get("rejected", [])]                    # (episode, reason)
        SOURCES.append(dict(path=path, created=c["created"], shard_roots=c["cfg"]["shard_roots"],
                            episodes={sp: len(v["eps"]) for sp, v in c["splits"].items()}))
        del c
    ids = [e["id"] for sp in SPLITS for e in EPS[sp]]
    assert len(ids) == len(set(ids)), "an episode appears in two caches"
    for sp in SPLITS:                                   # interleave the shards (training reads in order per epoch anyway)
        rng = np.random.default_rng([CFG["seed"], 77, SPLITS.index(sp)]); DATA[sp] = [DATA[sp][i] for i in rng.permutation(len(DATA[sp]))]
    CFG["merged_from"] = SOURCES
    for sp in SPLITS:
        print(f"{sp:>18}: {len(DATA[sp])} examples from {len(EPS[sp])} episodes, tasks " +
              str(dict(sorted(Counter(e['task_id'] for e in EPS[sp]).items()))))
else:
    print("not merging (merge_from is empty)")

## 7. Answer distributions and shortcut baselines

For each question: the accuracy of guessing the answer (majority answer of the train split) from
**nothing**, the **task family** only (what the spoken instruction gives), the **time** in the episode only
(0.5 s bins; what proprio gives under the fixed waypoint schedule), and **task + time**. A model should beat
these by a clear margin before we claim it uses perception. The same report is made for the canonical QA v1.

In [ ]:
def shortcut_table(train, tests, key_fns):
    """train/tests: lists of (question, answer, features dict). Majority answer per key on train -> accuracy per split."""
    rows = {}
    for q in sorted({x[0] for x in train}):
        tr = [x for x in train if x[0] == q]; row = {}
        for name, fn in key_fns.items():
            maj = {}
            for _, a, f in tr: maj.setdefault(fn(f), Counter())[a] += 1
            glob_maj = Counter(a for _, a, _ in tr).most_common(1)[0][0]
            for sp, te in tests.items():
                te_q = [x for x in te if x[0] == q]
                if not te_q: continue
                guess = [maj[fn(f)].most_common(1)[0][0] if fn(f) in maj else glob_maj for _, _, f in te_q]
                row[(sp, name)] = np.mean([g == a for g, (_, a, _) in zip(guess, te_q)])
        rows[q] = row
    return rows

def print_table(rows, title, names):
    sps = [sp for sp in SPLITS if any((sp, n) in r for r in rows.values() for n in names)]
    print(f"\n{title}\n{'':>14}" + "".join(f"{sp:>{11 * len(names)}}" for sp in sps))
    print(f"{'':>14}" + "".join(f"{n:>11}" for _ in sps for n in names))
    for q, r in rows.items():
        print(f"{q:>14}" + "".join(f"{r[(sp, n)]:11.2f}" if (sp, n) in r else f"{'-':>11}" for sp in sps for n in names))

KEYS = {"none": lambda f: 0, "task": lambda f: f["task"], "time": lambda f: f["tbin"], "task+time": lambda f: (f["task"], f["tbin"])}
feat = lambda ex: dict(task=ex["task_id"], tbin=ex["k"] // CFG["time_bin"])
for sp in SPLITS:
    if not DATA[sp]: continue
    print(f"\n{sp}: answer counts")
    for q in QUESTIONS:
        c = Counter(ex["a"] for ex in DATA[sp] if ex["task"] == q)
        if c: print(f"  {q:>12}: " + ", ".join(f"{a} {n}" for a, n in c.most_common()))
frame_rows = [(ex["task"], ex["a"], feat(ex)) for ex in DATA["train"]]
SHORTCUT_FRAME = shortcut_table(frame_rows, {sp: [(ex["task"], ex["a"], feat(ex)) for ex in DATA[sp]] for sp in SPLITS if sp != "train" and DATA[sp]}, KEYS)
print_table(SHORTCUT_FRAME, "frame-level questions: accuracy of the answer guessed from ... (train majority)", list(KEYS))

qa_rows = lambda sp: [(q["category"], str(q["expert_answer"]), dict(task=ep["task_id"], tbin=0)) for ep in EPS[sp] for q in ep["qa_v1"]]
SHORTCUT_V1 = shortcut_table(qa_rows("train"), {sp: qa_rows(sp) for sp in SPLITS if sp != "train" and EPS[sp]},
                             {"none": KEYS["none"], "task": KEYS["task"]})
print_table(SHORTCUT_V1, "canonical QA v1 (episode level): accuracy of the answer guessed from ...", ["none", "task"])

## 8. Save the cache

Format (loaded by `multirep_robot.ipynb` with `data="omni_v1"`):
```python
cache = pickle.load(open(path, "rb"))
train_data, train_eps = cache["splits"]["train"]["data"], cache["splits"]["train"]["eps"]
# example: task, q, a, ep (index into eps), k, views, points, point_rgb, proprio, speech, tactile, impact,
#          ee_delta (3,), grip_next, action_valid
# episode: id, split, task_id, task_name, instruction, color, start, goal, phase, q, finger, ee, obj, attached,
#          ee_delta, ee_target, grip_next, action_valid, mel, qa_v1, calib
```
Examples of the same frame share their arrays, so pickle stores every frame once.

In [ ]:
def tiny(v):
    return {k: (round(x, 3) if isinstance(x, float) else x) for k, x in v.items()}
CACHE = dict(version=CFG["version"], created=datetime.datetime.now().isoformat(timespec="seconds"), cfg=CFG,
             questions=QUESTIONS, tasks=TASKS, cams=CAMS, pc_center=PC_CENTER, pc_scale=PC_SCALE,
             splits={sp: dict(data=DATA[sp], eps=EPS[sp]) for sp in SPLITS},
             rejected=[(os.path.basename(d), why) for d, why in BAD],         # merged: (episode, reason) already
             shortcut_frame={q: {f"{sp}/{n}": float(v) for (sp, n), v in r.items()} for q, r in SHORTCUT_FRAME.items()},
             shortcut_v1={q: {f"{sp}/{n}": float(v) for (sp, n), v in r.items()} for q, r in SHORTCUT_V1.items()})
os.makedirs(os.path.dirname(os.path.abspath(CFG["out_path"])), exist_ok=True)
tmp = CFG["out_path"] + ".tmp"
with open(tmp, "wb") as f: pickle.dump(CACHE, f, protocol=pickle.HIGHEST_PROTOCOL)
os.replace(tmp, CFG["out_path"])
summary = dict(version=CFG["version"], created=CACHE["created"], bytes=os.path.getsize(CFG["out_path"]),
               episodes={sp: len(EPS[sp]) for sp in SPLITS}, examples={sp: len(DATA[sp]) for sp in SPLITS},
               rejected=len(BAD), shortcut_frame=CACHE["shortcut_frame"], shortcut_v1=CACHE["shortcut_v1"])
json.dump(summary, open(os.path.splitext(CFG["out_path"])[0] + "_summary.json", "w"), indent=1)
print(f"saved {CFG['out_path']} ({summary['bytes'] / 1e6:.0f} MB)", summary["examples"])

## 9. Look at one example per question

In [ ]:
import matplotlib.pyplot as plt
sp = next(s for s in SPLITS if DATA[s])
shown = [next(ex for ex in DATA[sp] if ex["task"] == q) for q in QUESTIONS if any(ex["task"] == q for ex in DATA[sp])]
fig, axs = plt.subplots(len(shown), 5, figsize=(15, 2.6 * len(shown)), dpi=90)
axs = np.atleast_2d(axs)
for row, ex in zip(axs, shown):
    ep = EPS[sp][ex["ep"]]
    for c in range(3):
        row[c].imshow(ex["views"][c]); row[c].axis("off"); row[c].set_title(CAMS[c], fontsize=8)
    row[0].set_title(f"{ep['id']} t={ex['k'] / FPS:.1f}s ({ep['phase'][ex['k']]})\n{CAMS[0]}", fontsize=8)
    P = ex["points"] * PC_SCALE + PC_CENTER
    row[3].scatter(P[:, 0], P[:, 1], c=ex["point_rgb"], s=2); row[3].set_aspect("equal"); row[3].set_title("points (top view, x-y)", fontsize=8)
    row[3].tick_params(labelsize=6)
    row[4].axis("off")
    row[4].text(0, 0.5, f"{ep['instruction']}\n\nQ: {ex['q']}\nA: {ex['a']}\n\nnext EE delta (cm): {np.round(ex['ee_delta'] * 100, 1)}"
                f"\nnext gripper: {ex['grip_next']:.3f} m", fontsize=8, va="center", wrap=True)
plt.tight_layout(); plt.show()